In [1]:
import pandas as pd
import numpy as np

# ==========================================
# ส่วนที่ 1: การตั้งค่าข้อมูล (Configuration)
# แก้ไขข้อมูลตรงนี้เพื่อ เพิ่ม/ลด สินค้า หรือ เดือน
# ==========================================

# 1.1 ข้อมูลสินค้า (Product Specs)
# เพิ่มสินค้าใหม่โดยใส่ Key เพิ่มใน Dict นี้
product_data = {
    'A': {'I0': 50,  'R': 8,  'Alloc_Pct': 12.5},
    'B': {'I0': 100, 'R': 10, 'Alloc_Pct': 25.0},
    'C': {'I0': 200, 'R': 12, 'Alloc_Pct': 50.0},
    'D': {'I0': 50,  'R': 6,  'Alloc_Pct': 12.5},
    # ตัวอย่างการเพิ่มสินค้า: 'E': {'I0': 0, 'R': 5, 'Alloc_Pct': 0},
}

# 1.2 ข้อมูลความต้องการลูกค้า (Forecast Demand)
# เพิ่มเดือนใหม่โดยเพิ่ม Key เดือน และใส่ยอดขายสินค้าให้ครบ
demand_data = {
    'ม.ค.': {'A': 400, 'B': 520, 'C': 600, 'D': 400},
    'ก.พ.': {'A': 350, 'B': 420, 'C': 500, 'D': 335},
    'มี.ค.': {'A': 260, 'B': 310, 'C': 360, 'D': 250},
    # ตัวอย่างการเพิ่มเดือน: 'เม.ย.': {'A': 300, 'B': 400, 'C': 400, 'D': 300},
}

# 1.3 ข้อมูลแผนรวม (Aggregate Plan)
# ต้องมีเดือนตรงกับ demand_data
aggregate_plan_data = {
    'ม.ค.': {'Target_Inv_Kg': 4050, 'Max_RM_Limit': 18500},
    'ก.พ.': {'Target_Inv_Kg': 3375, 'Max_RM_Limit': 18750},
    'มี.ค.': {'Target_Inv_Kg': 2475, 'Max_RM_Limit': 10000},
    # ตัวอย่างการเพิ่มเดือน: 'เม.ย.': {'Target_Inv_Kg': 2000, 'Max_RM_Limit': 15000},
}

# ==========================================
# ส่วนที่ 2: ระบบคำนวณอัตโนมัติ (Processing Logic)
# (ไม่ต้องแก้ไขส่วนนี้ ระบบจะรันตามข้อมูลข้างบน)
# ==========================================

# แปลงข้อมูลเป็น DataFrame เพื่อความง่ายในการคำนวณ
df_products = pd.DataFrame(product_data).T  # Index = Product Name
df_demand = pd.DataFrame(demand_data)       # Index = Product, Col = Month
df_agg = pd.DataFrame(aggregate_plan_data).T # Index = Month

products_list = df_products.index.tolist()
months_list = df_demand.columns.tolist()

# เตรียมตัวแปรเก็บผลลัพธ์
results = {
    'I': pd.DataFrame(index=products_list, columns=months_list),
    'D': pd.DataFrame(index=products_list, columns=months_list),
    'd': pd.DataFrame(index=products_list, columns=months_list)
}

# เริ่มต้น Inventory จาก I0 ของแต่ละสินค้า
current_inv = df_products['I0'].copy()

# ลูปคำนวณทีละเดือน
for month in months_list:
    # 1. ดึงค่าเป้าหมาย Inventory รวม (kg) ของเดือนนั้น
    target_kg = df_agg.loc[month, 'Target_Inv_Kg']
    
    # 2. คำนวณ I (Inventory ปลายงวดหน่วยสินค้า)
    # สูตร: (Target_Kg * %Alloc) / (100 * R)
    inv_t = (target_kg * (df_products['Alloc_Pct'] / 100)) / df_products['R']
    inv_t = inv_t.round().astype(int) # ปัดเศษตามโจทย์
    results['I'][month] = inv_t
    
    # 3. คำนวณ D (Net Requirement)
    # สูตร: FD - I_prev
    fd_t = df_demand[month]
    d_req_t = fd_t - current_inv
    results['D'][month] = d_req_t
    
    # 4. คำนวณ d (Production Plan)
    # สูตร Correction: d = D + I_current
    d_plan_t = d_req_t + inv_t
    results['d'][month] = d_plan_t
    
    # อัปเดต Inventory ต้นงวดรอบหน้า ให้เป็นปลายงวดรอบนี้
    current_inv = inv_t

# ==========================================
# ส่วนที่ 3: จัดรูปแบบตารางแสดงผล (Display Formatting)
# ==========================================

# สร้าง DataFrame รวมสำหรับแสดงผล
final_df = df_products[['I0', 'R']].copy()
final_df.columns = ['I0', 'Raw mat/unit'] # Rename เพื่อความสวยงาม

# รวมข้อมูล FD, D, d, I เข้าไป
for m in months_list: final_df[f'FD_{m}'] = df_demand[m]
for m in months_list: final_df[f'D_{m}'] = results['D'][m]
for m in months_list: final_df[f'd_{m}'] = results['d'][m]

final_df['%_Alloc'] = df_products['Alloc_Pct']
for m in months_list: final_df[f'I_{m}'] = results['I'][m]

# จัด Group Header (MultiIndex)
cols = [('Basic Info', 'I0'), ('Basic Info', 'Raw mat/unit')]
cols += [('ความต้องการ (FD)', m) for m in months_list]
cols += [('ต้องการผลิต (D)', m) for m in months_list]
cols += [('แผนการผลิตจริง (d)', m) for m in months_list]
cols += [('Inventory (I)', '%kg')]
cols += [('Inventory (I)', m) for m in months_list]

final_df.columns = pd.MultiIndex.from_tuples(cols)

# ==========================================
# ส่วนที่ 4: สรุปและตรวจสอบทรัพยากร (Summary Check)
# ==========================================

summary_rows = []
status_rows = []

for month in months_list:
    # คำนวณการใช้วัตถุดิบจริง: Sum(d * R)
    actual_d = results['d'][month]
    rm_per_unit = df_products['R']
    total_usage = (actual_d * rm_per_unit).sum()
    
    limit = df_agg.loc[month, 'Max_RM_Limit']
    
    summary_rows.append(total_usage)
    
    if total_usage > limit:
        diff = total_usage - limit
        status_rows.append(f"❌ เกิน (+{int(diff)})")
    else:
        status_rows.append("✅ ปกติ")

summary_df = pd.DataFrame({
    'แผนย่อย Heuristic (kg)': summary_rows,
    'แผนรวม Aggregate Limit (kg)': df_agg['Max_RM_Limit'].values,
    'สถานะ': status_rows
}, index=months_list).T

# แสดงผล
print(f"=== ผลลัพธ์การคำนวณ ({len(products_list)} Products x {len(months_list)} Months) ===")
display(final_df)
print("\n=== การตรวจสอบวัตถุดิบ (Resource Check) ===")
display(summary_df)

=== ผลลัพธ์การคำนวณ (4 Products x 3 Months) ===


Basic Info              ความต้องการ (FD)            ต้องการผลิต (D)       \
          I0 Raw mat/unit             ม.ค. ก.พ. มี.ค.            ม.ค. ก.พ.   
A       50.0          8.0              400  350   260           350.0  287   
B      100.0         10.0              520  420   310           420.0  319   
C      200.0         12.0              600  500   360           400.0  331   
D       50.0          6.0              400  335   250           350.0  251   

        แผนการผลิตจริง (d)            Inventory (I)                  
  มี.ค.               ม.ค. ก.พ. มี.ค.           %kg ม.ค. ก.พ. มี.ค.  
A   207              413.0  340   246          12.5   63   53    39  
B   226              521.0  403   288          25.0  101   84    62  
C   219              569.0  472   322          50.0  169  141   103  
D   180              434.0  321   232          12.5   84   70    52


=== การตรวจสอบวัตถุดิบ (Resource Check) ===


,ม.ค.,ก.พ.,มี.ค.
แผนย่อย Heuristic (kg),17946.0,14340.0,10104.0
แผนรวม Aggregate Limit (kg),18500,18750,10000
สถานะ,✅ ปกติ,✅ ปกติ,❌ เกิน (+104)


In [2]:
import pandas as pd
import numpy as np

# ==========================================
# ส่วนที่ 1: การตั้งค่าข้อมูล (Configuration)
# แก้ไขตัวเลขตรงนี้เพื่อเปลี่ยนโจทย์
# ==========================================

# 1.1 ข้อมูลทรัพยากรการผลิต (Global Parameters)
total_labor_hours = 1000   # ชั่วโมงแรงงานรวมที่มี (ชั่วโมง)
setup_time_per_model = 3   # เวลาเปลี่ยนรุ่นการผลิต (ชั่วโมง/ครั้ง)
prod_time_per_unit = 2     # เวลาผลิตต่อหน่วย (ชั่วโมง/ตัว)
ss_percentage = 25         # % Safety Stock ของความต้องการเฉลี่ย (ใส่เป็นตัวเลข เช่น 25 คือ 25%)

# 1.2 ข้อมูลสินค้า (Product Data)
# Format: 'ชื่อรุ่น': {'I0': สินค้าคงคลังเริ่มต้น, 'Demand_Avg': ความต้องการเฉลี่ยต่อสัปดาห์}
product_data = {
    'Model 1': {'I0': 0,  'Demand_Avg': 10},
    'Model 2': {'I0': 30, 'Demand_Avg': 40},
    'Model 3': {'I0': 80, 'Demand_Avg': 60},
    'Model 4': {'I0': 25, 'Demand_Avg': 90},
}

# ==========================================
# ส่วนที่ 2: ระบบคำนวณ (Calculation Engine)
# ==========================================

# สร้าง DataFrame จากข้อมูลดิบ
df = pd.DataFrame(product_data).T

# 2.1 คำนวณ Safety Stock (SS)
# สูตร: SS = Demand_Avg * (ss_percentage / 100)
df['SS'] = df['Demand_Avg'] * (ss_percentage / 100)

# 2.2 คำนวณกำลังการผลิตรวม (Pt)
num_models = len(df)
total_setup_time = num_models * setup_time_per_model
available_prod_time = total_labor_hours - total_setup_time

# Pt = เวลาผลิตจริง / เวลาต่อหน่วย
Pt = available_prod_time / prod_time_per_unit

# 2.3 คำนวณตัวแปรสำหรับสมการ (4.16)
# เทอมรวม: Sum(I_prev - SS)
sum_diff_inv_ss = (df['I0'] - df['SS']).sum()

# เทอมเป้าหมายรวม: Pt + Sum(I_prev - SS) -> (ค่า 579 ในตัวอย่าง)
aggregate_target = Pt + sum_diff_inv_ss

# ผลรวมความต้องการเฉลี่ย Sum(D_it)
sum_demand = df['Demand_Avg'].sum()

# 2.4 คำนวณปริมาณการผลิต (Q_it) และ Run-out Time (r_it)
# สูตร Q_it = [Aggregate_Target] * (D_it / Sum_D) + SS_it - I_prev
# หมายเหตุ: ในรูปภาพมีการปัดเศษ Q เป็นจำนวนเต็ม ผมจึงใช้ round()

df['Q_calc'] = (aggregate_target * (df['Demand_Avg'] / sum_demand)) + df['SS'] - df['I0']
df['Q_final'] = df['Q_calc'].round().astype(int) # ปัดเป็นจำนวนเต็มเพื่อผลิตจริง

# สูตร Run-out Time (r_it) = (Q_it + I_prev - SS_it) / D_it
# ใช้ค่า Q_final (จำนวนเต็ม) ในการคำนวณตามหลักปฏิบัติ
df['Run_out_Time (r)'] = (df['Q_final'] + df['I0'] - df['SS']) / df['Demand_Avg']

# ==========================================
# ส่วนที่ 3: แสดงผลลัพธ์ (Display)
# ==========================================

# จัดรูปแบบตารางให้สวยงามเหมือนในรูป
output_df = df[['I0', 'Demand_Avg', 'SS', 'Q_final', 'Run_out_Time (r)']].copy()
output_df.columns = [
    'Inventory Init (I0)', 
    'Avg Demand (D)', 
    'Safety Stock (SS)', 
    'Production Qty (Q)', 
    'Run-out Time (Weeks)'
]

print(f"=== ข้อมูลสรุปทรัพยากร ===")
print(f"Total Hours: {total_labor_hours} hrs")
print(f"Total Setup: {total_setup_time} hrs ({num_models} models x {setup_time_per_model} hrs)")
print(f"Available for Prod: {available_prod_time} hrs")
print(f"Production Capacity (Pt): {Pt} units")
print(f"Aggregate Target Base (Pt + Sum(I-SS)): {aggregate_target} units")
print("-" * 30)
print(f"\n=== ตารางผลลัพธ์ (เหมือนภาพที่ 2) ===")
display(output_df)

# ตรวจสอบความถูกต้อง (Check Total Q vs Pt)
# ผลรวม Q ควรใกล้เคียง Pt (อาจต่างกันเล็กน้อยจากการปัดเศษ)
total_Q = output_df['Production Qty (Q)'].sum()
print(f"\nCheck: Total Production Q ({total_Q}) vs Capacity Pt ({Pt})")

=== ข้อมูลสรุปทรัพยากร ===
Total Hours: 1000 hrs
Total Setup: 12 hrs (4 models x 3 hrs)
Available for Prod: 988 hrs
Production Capacity (Pt): 494.0 units
Aggregate Target Base (Pt + Sum(I-SS)): 579.0 units
------------------------------

=== ตารางผลลัพธ์ (เหมือนภาพที่ 2) ===


,Inventory Init (I0),Avg Demand (D),Safety Stock (SS),Production Qty (Q),Run-out Time (Weeks)
Model 1,0,10,2.5,31,2.850000
Model 2,30,40,10.0,96,2.900000
Model 3,80,60,15.0,109,2.900000
Model 4,25,90,22.5,258,2.894444



Check: Total Production Q (494) vs Capacity Pt (494.0)


# ตย.

In [1]:
import pandas as pd
import numpy as np

# ==========================================
# ส่วนที่ 1: การตั้งค่าข้อมูล (Configuration)
# แก้ไขข้อมูลตรงนี้เพื่อ เพิ่ม/ลด สินค้า หรือ เดือน
# ==========================================

# 1.1 ข้อมูลสินค้า (Product Specs)
# เพิ่มสินค้าใหม่โดยใส่ Key เพิ่มใน Dict นี้
product_data = {
    'A': {'I0': 90,  'R': 6,  'Alloc_Pct': 30},
    'B': {'I0': 100, 'R': 8, 'Alloc_Pct': 40},
    'C': {'I0': 150, 'R': 5, 'Alloc_Pct': 30}
    # ตัวอย่างการเพิ่มสินค้า: 'E': {'I0': 0, 'R': 5, 'Alloc_Pct': 0},
}

# 1.2 ข้อมูลความต้องการลูกค้า (Forecast Demand)
# เพิ่มเดือนใหม่โดยเพิ่ม Key เดือน และใส่ยอดขายสินค้าให้ครบ
demand_data = {
    'ม.ค.': {'A': 1200, 'B': 800, 'C': 1000},
    'ก.พ.': {'A': 1500, 'B': 950, 'C': 700},
    'มี.ค.': {'A': 1600, 'B': 750, 'C': 1100},
    'เม.ย.': {'A': 1400, 'B': 850, 'C': 750},
    # ตัวอย่างการเพิ่มเดือน: 'เม.ย.': {'A': 300, 'B': 400, 'C': 400, 'D': 300},
}

# 1.3 ข้อมูลแผนรวม (Aggregate Plan)
# ต้องมีเดือนตรงกับ demand_data
aggregate_plan_data = {
    'ม.ค.': {'Target_Inv_Kg': 2000, 'Max_RM_Limit': 18500},
    'ก.พ.': {'Target_Inv_Kg': 2100, 'Max_RM_Limit': 18750},
    'มี.ค.': {'Target_Inv_Kg': 2200, 'Max_RM_Limit': 10000},
    'เม.ย.': {'Target_Inv_Kg': 2400, 'Max_RM_Limit': 15000}
    # ตัวอย่างการเพิ่มเดือน: 'เม.ย.': {'Target_Inv_Kg': 2000, 'Max_RM_Limit': 15000},
}

# ==========================================
# ส่วนที่ 2: ระบบคำนวณอัตโนมัติ (Processing Logic)
# (ไม่ต้องแก้ไขส่วนนี้ ระบบจะรันตามข้อมูลข้างบน)
# ==========================================

# แปลงข้อมูลเป็น DataFrame เพื่อความง่ายในการคำนวณ
df_products = pd.DataFrame(product_data).T  # Index = Product Name
df_demand = pd.DataFrame(demand_data)       # Index = Product, Col = Month
df_agg = pd.DataFrame(aggregate_plan_data).T # Index = Month

products_list = df_products.index.tolist()
months_list = df_demand.columns.tolist()

# เตรียมตัวแปรเก็บผลลัพธ์
results = {
    'I': pd.DataFrame(index=products_list, columns=months_list),
    'D': pd.DataFrame(index=products_list, columns=months_list),
    'd': pd.DataFrame(index=products_list, columns=months_list)
}

# เริ่มต้น Inventory จาก I0 ของแต่ละสินค้า
current_inv = df_products['I0'].copy()

# ลูปคำนวณทีละเดือน
for month in months_list:
    # 1. ดึงค่าเป้าหมาย Inventory รวม (kg) ของเดือนนั้น
    target_kg = df_agg.loc[month, 'Target_Inv_Kg']
    
    # 2. คำนวณ I (Inventory ปลายงวดหน่วยสินค้า)
    # สูตร: (Target_Kg * %Alloc) / (100 * R)
    inv_t = (target_kg * (df_products['Alloc_Pct'] / 100)) / df_products['R']
    inv_t = inv_t.round().astype(int) # ปัดเศษตามโจทย์
    results['I'][month] = inv_t
    
    # 3. คำนวณ D (Net Requirement)
    # สูตร: FD - I_prev
    fd_t = df_demand[month]
    d_req_t = fd_t - current_inv
    results['D'][month] = d_req_t
    
    # 4. คำนวณ d (Production Plan)
    # สูตร Correction: d = D + I_current
    d_plan_t = d_req_t + inv_t
    results['d'][month] = d_plan_t
    
    # อัปเดต Inventory ต้นงวดรอบหน้า ให้เป็นปลายงวดรอบนี้
    current_inv = inv_t

# ==========================================
# ส่วนที่ 3: จัดรูปแบบตารางแสดงผล (Display Formatting)
# ==========================================

# สร้าง DataFrame รวมสำหรับแสดงผล
final_df = df_products[['I0', 'R']].copy()
final_df.columns = ['I0', 'Raw mat/unit'] # Rename เพื่อความสวยงาม

# รวมข้อมูล FD, D, d, I เข้าไป
for m in months_list: final_df[f'FD_{m}'] = df_demand[m]
for m in months_list: final_df[f'D_{m}'] = results['D'][m]
for m in months_list: final_df[f'd_{m}'] = results['d'][m]

final_df['%_Alloc'] = df_products['Alloc_Pct']
for m in months_list: final_df[f'I_{m}'] = results['I'][m]

# จัด Group Header (MultiIndex)
cols = [('Basic Info', 'I0'), ('Basic Info', 'Raw mat/unit')]
cols += [('ความต้องการ (FD)', m) for m in months_list]
cols += [('ต้องการผลิต (D)', m) for m in months_list]
cols += [('แผนการผลิตจริง (d)', m) for m in months_list]
cols += [('Inventory (I)', '%kg')]
cols += [('Inventory (I)', m) for m in months_list]

final_df.columns = pd.MultiIndex.from_tuples(cols)

# ==========================================
# ส่วนที่ 4: สรุปและตรวจสอบทรัพยากร (Summary Check)
# ==========================================

summary_rows = []
status_rows = []

for month in months_list:
    # คำนวณการใช้วัตถุดิบจริง: Sum(d * R)
    actual_d = results['d'][month]
    rm_per_unit = df_products['R']
    total_usage = (actual_d * rm_per_unit).sum()
    
    limit = df_agg.loc[month, 'Max_RM_Limit']
    
    summary_rows.append(total_usage)
    
    if total_usage > limit:
        diff = total_usage - limit
        status_rows.append(f"❌ เกิน (+{int(diff)})")
    else:
        status_rows.append("✅ ปกติ")

summary_df = pd.DataFrame({
    'แผนย่อย Heuristic (kg)': summary_rows,
    'แผนรวม Aggregate Limit (kg)': df_agg['Max_RM_Limit'].values,
    'สถานะ': status_rows
}, index=months_list).T

# แสดงผล
print(f"=== ผลลัพธ์การคำนวณ ({len(products_list)} Products x {len(months_list)} Months) ===")
display(final_df)
print("\n=== การตรวจสอบวัตถุดิบ (Resource Check) ===")
display(summary_df)

=== ผลลัพธ์การคำนวณ (3 Products x 4 Months) ===


Basic Info              ความต้องการ (FD)                   ต้องการผลิต (D)  \
          I0 Raw mat/unit             ม.ค.  ก.พ. มี.ค. เม.ย.            ม.ค.   
A         90            6             1200  1500  1600  1400            1110   
B        100            8              800   950   750   850             700   
C        150            5             1000   700  1100   750             850   

                    แผนการผลิตจริง (d)                   Inventory (I)       \
   ก.พ. มี.ค. เม.ย.               ม.ค.  ก.พ. มี.ค. เม.ย.           %kg ม.ค.   
A  1400  1495  1290               1210  1505  1605  1410            30  100   
B   850   645   740                800   955   755   860            40  100   
C   580   974   618                970   706  1106   762            30  120   

                    
  ก.พ. มี.ค. เม.ย.  
A  105   110   120  
B  105   110   120  
C  126   132   144


=== การตรวจสอบวัตถุดิบ (Resource Check) ===


,ม.ค.,ก.พ.,มี.ค.,เม.ย.
แผนย่อย Heuristic (kg),18510,20200,21200,19150
แผนรวม Aggregate Limit (kg),18500,18750,10000,15000
สถานะ,❌ เกิน (+10),❌ เกิน (+1450),❌ เกิน (+11200),❌ เกิน (+4150)


In [ ]:
data = {
    'Model':       ['A', 'B', 'C', 'D'],
    'Inventory':   [5, 7, 10, 15],       # Stock เดิมที่มี (ชิ้น)
    'Demand':      [34, 70, 130, 150],   # ความต้องการ (ชิ้น)
    'Prod_Rate':   [0.5, 1.5, 2.0, 2.5], # เวลาผลิต (ชม./ชิ้น)
    'Setup_Time':  [3, 2, 4, 3]          # เวลา Setup (ชม.)
}


In [17]:
import pandas as pd

# ==========================================
# 1. SETUP DATA (ส่วนแก้ไขข้อมูล)
# ==========================================
GROSS_CAPACITY_HOURS = 760 

# กำหนด % Safety Stock ที่ต้องการ (เช่น 0.20 คือ 20% ของ Demand)
SAFETY_STOCK_RATIO = 0.12  

data = {
    'Model':       ['A', 'B', 'C', 'D'],
    'Inventory':   [5, 7, 10, 15],       # Stock เดิมที่มี (ชิ้น)
    'Demand':      [34, 70, 130, 150],   # ความต้องการ (ชิ้น)
    'Prod_Rate':   [0.5, 1.5, 2.0, 2.5], # เวลาผลิต (ชม./ชิ้น)
    'Setup_Time':  [3, 2, 4, 3]          # เวลา Setup (ชม.)
}

df = pd.DataFrame(data)

# ==========================================
# 2. PREPARATION & CALCULATION
# ==========================================

# 2.1 คำนวณ Safety Stock (ชิ้น) จาก % ของ Demand
df['Safety_Stock'] = df['Demand'] * SAFETY_STOCK_RATIO

# 2.2 แปลงทุกอย่างเป็นหน่วย "ชั่วโมง" (Resource Units)
df['D_hrs']  = df['Demand'] * df['Prod_Rate']       # D_it
df['I_hrs']  = df['Inventory'] * df['Prod_Rate']    # I_i,t-1
df['SS_hrs'] = df['Safety_Stock'] * df['Prod_Rate'] # SS_it

# 2.3 คำนวณตัวแปรสำหรับสูตร 4.16
# Pt (กำลังการผลิตจริง)
Pt = GROSS_CAPACITY_HOURS - df['Setup_Time'].sum()

# เทอมผลรวม (Summation Terms)
sum_D_hrs = df['D_hrs'].sum()                       
sum_Net_Inv_hrs = (df['I_hrs'] - df['SS_hrs']).sum() # sum(I - SS)

# 2.4 คำนวณตามสูตรสมการที่ 4.16
# ส่วนที่ 1: ทรัพยากรคงเหลือสุทธิ (Net Resources Available)
total_available_resource = Pt + sum_Net_Inv_hrs

# ส่วนที่ 2: สัดส่วนความต้องการ (Demand Ratio)
demand_ratio = df['D_hrs'] / sum_D_hrs

# ส่วนที่ 3: คำนวณ Q (หน่วยชั่วโมง)
# สูตร: Q = [ (Pt + sum(I-SS)) * (D_i / sum(D)) ] + SS_i - I_i
df['Q_hours'] = (total_available_resource * demand_ratio) + df['SS_hrs'] - df['I_hrs']

# 2.5 แปลงกลับเป็นจำนวนชิ้น (Units)
df['Production_Qty'] = df['Q_hours'] / df['Prod_Rate']

# จัดรูปแบบตัวเลขให้สวยงาม (ปัดเศษ)
df['Production_Qty'] = df['Production_Qty'].round(2)
df['Safety_Stock'] = df['Safety_Stock'].round(2) # แสดง SS ที่คำนวณได้
df['Q_hours'] = df['Q_hours'].round(2)

# ==========================================
# 3. DISPLAY RESULTS
# ==========================================
print(f"--- ผลการคำนวณ (Safety Stock = {SAFETY_STOCK_RATIO*100}% ของ Demand) ---")
print(f"Pt (Net Capacity): {Pt} ชม.")
print(f"Sum(Demand Hrs): {sum_D_hrs:.2f} ชม.")
print("-" * 60)

output_df = df[['Model', 'Inventory', 'Demand', 'Safety_Stock', 'Production_Qty', 'Q_hours']].copy()
output_df.columns = ['Model', 'Stock เดิม', 'Demand', f'SS ({int(SAFETY_STOCK_RATIO*100)}%)', 'ผลิตเพิ่ม (Q)', 'เวลาที่ใช้ (ชม.)']

from IPython.display import display
display(output_df)

# ตรวจสอบความถูกต้อง
total_used = output_df['เวลาที่ใช้ (ชม.)'].sum()
print(f"\nรวมเวลาที่ใช้ผลิตทั้งหมด: {total_used:.2f} ชม. (Target Pt: {Pt})")

--- ผลการคำนวณ (Safety Stock = 12.0% ของ Demand) ---
Pt (Net Capacity): 748 ชม.
Sum(Demand Hrs): 757.00 ชม.
------------------------------------------------------------


,Model,Stock เดิม,Demand,SS (12%),ผลิตเพิ่ม (Q),เวลาที่ใช้ (ชม.)
0,A,5,34,4.08,31.76,15.88
1,B,7,70,8.40,68.69,103.03
2,C,10,130,15.60,130.56,261.12
3,D,15,150,18.00,147.19,367.97



รวมเวลาที่ใช้ผลิตทั้งหมด: 748.00 ชม. (Target Pt: 748)


# ดูหน่วยดี ๆ

In [16]:
import pandas as pd

# ==========================================
# 1. SETUP DATA (ส่วนแก้ไขข้อมูล)
# ==========================================
GROSS_CAPACITY_HOURS = 600

# กำหนด % Safety Stock ที่ต้องการ (เช่น 0.25 คือ 25% ของ Demand)
SAFETY_STOCK_RATIO = 0.1

data = {
    'Model':       ['A', 'B', 'C', 'D'],
    'Inventory':   [0, 6, 14, 50],      # Stock เดิมที่มี (I_i,t-1)
    'Demand':      [20, 80, 140, 150],     # ความต้องการ (D_it)
    'Prod_Rate':   [2, 0.5, 0.5, 1],         # เวลาผลิต (ชม./ชิ้น)
    'Setup_Time':  [4, 3, 3, 2]          # เวลา Setup (ชม.)
}

df = pd.DataFrame(data)

# ==========================================
# 2. PREPARATION & CALCULATION
# ==========================================

# 2.1 คำนวณ Safety Stock (ชิ้น) จาก % ของ Demand
df['Safety_Stock'] = df['Demand'] * SAFETY_STOCK_RATIO

# 2.2 แปลงทุกอย่างเป็นหน่วย "ชั่วโมง" (Resource Units)
df['D_hrs']  = df['Demand'] * df['Prod_Rate']       # D_it (in hours)
df['I_hrs']  = df['Inventory'] * df['Prod_Rate']    # I_i,t-1 (in hours)
df['SS_hrs'] = df['Safety_Stock'] * df['Prod_Rate'] # SS_it (in hours)

# 2.3 คำนวณตัวแปรสำหรับสูตร 4.16
# Pt (กำลังการผลิตจริง)
Pt = GROSS_CAPACITY_HOURS - df['Setup_Time'].sum()

# เทอมผลรวม (Summation Terms)
sum_D_hrs = df['D_hrs'].sum()                   
sum_Net_Inv_hrs = (df['I_hrs'] - df['SS_hrs']).sum() # sum(I - SS)

# 2.4 คำนวณปริมาณการผลิต (Q) ตามสูตรสมการที่ 4.16
# ส่วนที่ 1: ทรัพยากรคงเหลือสุทธิ (Net Resources Available)
total_available_resource = Pt + sum_Net_Inv_hrs

# ส่วนที่ 2: สัดส่วนความต้องการ (Demand Ratio)
demand_ratio = df['D_hrs'] / sum_D_hrs

# ส่วนที่ 3: คำนวณ Q (หน่วยชั่วโมง)
# สูตร: Q = [ (Pt + sum(I-SS)) * (D_i / sum(D)) ] + SS_i - I_i
df['Q_hours'] = (total_available_resource * demand_ratio) + df['SS_hrs'] - df['I_hrs']

# 2.5 แปลงกลับเป็นจำนวนชิ้น (Units)
df['Production_Qty'] = df['Q_hours'] / df['Prod_Rate']

# ==========================================
# [NEW] 2.6 คำนวณ Run-out Time (r) ตามสมการที่ 4.15
# ==========================================
# สูตร: r = (Q + I - SS) / D
# หมายเหตุ: จะใช้หน่วย "ชิ้น" หรือ "ชั่วโมง" คำนวณก็ได้ (ผลลัพธ์เท่ากัน) ในที่นี้ใช้หน่วยชิ้น
df['Run_out_Time'] = (df['Production_Qty'] + df['Inventory'] - df['Safety_Stock']) / df['Demand']


# จัดรูปแบบตัวเลขให้สวยงาม (ปัดเศษ)
df['Production_Qty'] = df['Production_Qty'].round(2)
df['Safety_Stock'] = df['Safety_Stock'].round(2)
df['Q_hours'] = df['Q_hours'].round(2)
df['Run_out_Time'] = df['Run_out_Time'].round(4) # ทศนิยม 4 ตำแหน่งเพื่อดูความละเอียด

# ==========================================
# 3. DISPLAY RESULTS
# ==========================================
print(f"--- ผลการคำนวณ (Safety Stock = {SAFETY_STOCK_RATIO*100}% ของ Demand) ---")
print(f"Pt (Net Capacity): {Pt} ชม.")
print(f"Sum(Demand Hrs): {sum_D_hrs:.2f} ชม.")
print("-" * 80)

output_df = df[['Model', 'Inventory', 'Demand', 'Safety_Stock', 'Production_Qty', 'Run_out_Time', 'Q_hours']].copy()
# เปลี่ยนชื่อคอลัมน์สำหรับแสดงผล
output_df.columns = [
    'Model', 
    'Stock เดิม (I)', 
    'Demand (D)', 
    'SS', 
    'ผลิตเพิ่ม (Q)', 
    'Run-out Time (r)', # แสดงค่า r ที่คำนวณได้
    'เวลาผลิต (ชม.)'
]

from IPython.display import display
display(output_df)

# ตรวจสอบความถูกต้อง
total_used = output_df['เวลาผลิต (ชม.)'].sum()
print(f"\nรวมเวลาที่ใช้ผลิตทั้งหมด: {total_used:.2f} ชม. (Target Pt: {Pt})")
print("\n*หมายเหตุ: ตามทฤษฎี Run-out Time Model ค่า r ของทุกสินค้าควรจะเท่ากัน (หรือใกล้เคียงกันมาก)")

--- ผลการคำนวณ (Safety Stock = 10.0% ของ Demand) ---
Pt (Net Capacity): 588 ชม.
Sum(Demand Hrs): 300.00 ชม.
--------------------------------------------------------------------------------


,Model,Stock เดิม (I),Demand (D),SS,ผลิตเพิ่ม (Q),Run-out Time (r),เวลาผลิต (ชม.)
0,A,0,20,2.0,43.2,2.06,86.4
1,B,6,80,8.0,166.8,2.06,83.4
2,C,14,140,14.0,288.4,2.06,144.2
3,D,50,150,15.0,274.0,2.06,274.0



รวมเวลาที่ใช้ผลิตทั้งหมด: 588.00 ชม. (Target Pt: 588)

*หมายเหตุ: ตามทฤษฎี Run-out Time Model ค่า r ของทุกสินค้าควรจะเท่ากัน (หรือใกล้เคียงกันมาก)


In [ ]:
data = {
    'Model':       ['A', 'B', 'C', 'D'],
    'Inventory':   [5, 7, 10, 15],       # Stock เดิมที่มี (ชิ้น)
    'Demand':      [34, 70, 130, 150],   # ความต้องการ (ชิ้น)
    'Prod_Rate':   [0.5, 1.5, 2.0, 2.5], # เวลาผลิต (ชม./ชิ้น)
    'Setup_Time':  [3, 2, 4, 3]          # เวลา Setup (ชม.)
}

In [19]:
import pandas as pd

# ==========================================
# 1. SETUP DATA (ส่วนแก้ไขข้อมูล)
# ==========================================
GROSS_CAPACITY_HOURS = 760

# กำหนด % Safety Stock ที่ต้องการ (เช่น 0.25 คือ 25% ของ Demand)
SAFETY_STOCK_RATIO = 0.12

data = {
    'Model':       ['A', 'B', 'C', 'D'],
    'Inventory':   [5, 7, 10, 15],       # Stock เดิมที่มี (ชิ้น)
    'Demand':      [34, 70, 130, 150],   # ความต้องการ (ชิ้น)
    'Prod_Rate':   [0.5, 1.5, 2.0, 2.5], # เวลาผลิต (ชม./ชิ้น)
    'Setup_Time':  [3, 2, 4, 3]          # เวลา Setup (ชม.)
}

df = pd.DataFrame(data)

# ==========================================
# 2. PREPARATION & CALCULATION
# ==========================================

# 2.1 คำนวณ Safety Stock (ชิ้น) จาก % ของ Demand
df['Safety_Stock'] = df['Demand'] * SAFETY_STOCK_RATIO

# 2.2 แปลงทุกอย่างเป็นหน่วย "ชั่วโมง" (Resource Units)
df['D_hrs']  = df['Demand'] * df['Prod_Rate']       # D_it (in hours)
df['I_hrs']  = df['Inventory'] * df['Prod_Rate']    # I_i,t-1 (in hours)
df['SS_hrs'] = df['Safety_Stock'] * df['Prod_Rate'] # SS_it (in hours)

# 2.3 คำนวณตัวแปรสำหรับสูตร 4.16
# Pt (กำลังการผลิตจริง)
Pt = GROSS_CAPACITY_HOURS - df['Setup_Time'].sum()

# เทอมผลรวม (Summation Terms)
sum_D_hrs = df['D_hrs'].sum()                   
sum_Net_Inv_hrs = (df['I_hrs'] - df['SS_hrs']).sum() # sum(I - SS)

# 2.4 คำนวณปริมาณการผลิต (Q) ตามสูตรสมการที่ 4.16
# ส่วนที่ 1: ทรัพยากรคงเหลือสุทธิ (Net Resources Available)
total_available_resource = Pt + sum_Net_Inv_hrs

# ส่วนที่ 2: สัดส่วนความต้องการ (Demand Ratio)
demand_ratio = df['D_hrs'] / sum_D_hrs

# ส่วนที่ 3: คำนวณ Q (หน่วยชั่วโมง)
# สูตร: Q = [ (Pt + sum(I-SS)) * (D_i / sum(D)) ] + SS_i - I_i
df['Q_hours'] = (total_available_resource * demand_ratio) + df['SS_hrs'] - df['I_hrs']

# 2.5 แปลงกลับเป็นจำนวนชิ้น (Units)
df['Production_Qty'] = df['Q_hours'] / df['Prod_Rate']

# ==========================================
# [NEW] 2.6 คำนวณ Run-out Time (r) ตามสมการที่ 4.15
# ==========================================
# สูตร: r = (Q + I - SS) / D
# หมายเหตุ: จะใช้หน่วย "ชิ้น" หรือ "ชั่วโมง" คำนวณก็ได้ (ผลลัพธ์เท่ากัน) ในที่นี้ใช้หน่วยชิ้น
df['Run_out_Time'] = (df['Production_Qty'] + df['Inventory'] - df['Safety_Stock']) / df['Demand']


# จัดรูปแบบตัวเลขให้สวยงาม (ปัดเศษ)
df['Production_Qty'] = df['Production_Qty'].round(2)
df['Safety_Stock'] = df['Safety_Stock'].round(2)
df['Q_hours'] = df['Q_hours'].round(2)
df['Run_out_Time'] = df['Run_out_Time'].round(4) # ทศนิยม 4 ตำแหน่งเพื่อดูความละเอียด

# ==========================================
# 3. DISPLAY RESULTS
# ==========================================
print(f"--- ผลการคำนวณ (Safety Stock = {SAFETY_STOCK_RATIO*100}% ของ Demand) ---")
print(f"Pt (Net Capacity): {Pt} ชม.")
print(f"Sum(Demand Hrs): {sum_D_hrs:.2f} ชม.")
print("-" * 80)

output_df = df[['Model', 'Inventory', 'Demand', 'Safety_Stock', 'Production_Qty', 'Run_out_Time', 'Q_hours']].copy()
# เปลี่ยนชื่อคอลัมน์สำหรับแสดงผล
output_df.columns = [
    'Model', 
    'Stock เดิม (I)', 
    'Demand (D)', 
    'SS', 
    'ผลิตเพิ่ม (Q)', 
    'Run-out Time (r)', # แสดงค่า r ที่คำนวณได้
    'เวลาผลิต (ชม.)'
]

from IPython.display import display
display(output_df)

# ตรวจสอบความถูกต้อง
total_used = output_df['เวลาผลิต (ชม.)'].sum()
print(f"\nรวมเวลาที่ใช้ผลิตทั้งหมด: {total_used:.2f} ชม. (Target Pt: {Pt})")
print("\n*หมายเหตุ: ตามทฤษฎี Run-out Time Model ค่า r ของทุกสินค้าควรจะเท่ากัน (หรือใกล้เคียงกันมาก)")

--- ผลการคำนวณ (Safety Stock = 12.0% ของ Demand) ---
Pt (Net Capacity): 748 ชม.
Sum(Demand Hrs): 757.00 ชม.
--------------------------------------------------------------------------------


,Model,Stock เดิม (I),Demand (D),SS,ผลิตเพิ่ม (Q),Run-out Time (r),เวลาผลิต (ชม.)
0,A,5,34,4.08,31.76,0.9612,15.88
1,B,7,70,8.40,68.69,0.9612,103.03
2,C,10,130,15.60,130.56,0.9612,261.12
3,D,15,150,18.00,147.19,0.9612,367.97



รวมเวลาที่ใช้ผลิตทั้งหมด: 748.00 ชม. (Target Pt: 748)

*หมายเหตุ: ตามทฤษฎี Run-out Time Model ค่า r ของทุกสินค้าควรจะเท่ากัน (หรือใกล้เคียงกันมาก)
